# Notebook 1: Re-validate baseline, confusion matrices, per-image predictions
EAAI-26-20775 revision, Day 1.

**What this does** (about 20 minutes on a T4, no training):
1. Unzips `dataset_clean.zip` and checks it has 1,206 train and 300 val images.
2. Loads your baseline `best.pt` and prints the Ultralytics version it was trained with, plus its training settings.
3. Validates at 768 and at 640 input size (Reviewer 2, comment 6: 768/768 and 768/640).
4. Builds confusion matrices at confidence 0.25 and at the deployed 0.60 (Reviewer 3, comment 10).
5. Saves every prediction on the 300 validation images, for the threshold analysis (Reviewer 3.3, Reviewer 4.10).

Everything is saved to `MyDrive/fabric/revision/nb1/`. Nothing in your existing folders is changed.

**Before running:** Runtime → Change runtime type → T4 GPU. Then Runtime → Run all.

In [ ]:
# ---- Settings (only change these if your Drive paths differ) ----
DRIVE_ROOT = "/content/drive/MyDrive/fabric"
DATASET_ZIP = f"{DRIVE_ROOT}/dataset_clean.zip"
BASELINE_RUN = f"{DRIVE_ROOT}/runs_clean/detect/fabric_defect_yolov8n_clean"
BEST_PT = f"{BASELINE_RUN}/weights/best.pt"
OUT_DIR = f"{DRIVE_ROOT}/revision/nb1"
WORK = "/content/data"          # local, fast disk
DEVICE = 0                      # GPU
LOCAL_TEST = False              # leave False on Colab

In [ ]:
import os, sys, subprocess
if not LOCAL_TEST:
    from google.colab import drive
    drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)
for p in (DATASET_ZIP, BEST_PT):
    assert os.path.exists(p), f"Not found: {p}  (check the path in the Settings cell)"
os.makedirs(OUT_DIR, exist_ok=True)
print("Inputs found. Outputs will go to", OUT_DIR)

In [ ]:
# ---- 1. Unzip dataset and check counts ----
import zipfile, glob
if not glob.glob(f"{WORK}/**/images/train", recursive=True):
    with zipfile.ZipFile(DATASET_ZIP) as z:
        z.extractall(WORK)
root = os.path.dirname(os.path.dirname(sorted(glob.glob(f"{WORK}/**/images/train", recursive=True))[0]))
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp")
counts = {s: len([f for f in os.listdir(f"{root}/images/{s}") if f.lower().endswith(IMG_EXT)]) for s in ("train", "val")}
print("Dataset root:", root)
print("Image counts:", counts, "(expected train 1206, val 300)")
if counts != {"train": 1206, "val": 300}:
    print("WARNING: counts differ from the manuscript. Stop and report this before continuing.")

In [ ]:
# ---- 2. Load baseline, print version and training settings ----
import json, yaml, torch, ultralytics
from ultralytics import YOLO
ckpt = torch.load(BEST_PT, map_location="cpu", weights_only=False)
info = {
    "ultralytics_now": ultralytics.__version__,
    "ultralytics_trained_with": ckpt.get("version"),
    "best_epoch": ckpt.get("epoch"),
    "train_args_in_checkpoint": ckpt.get("train_args"),
}
args_yaml = f"{BASELINE_RUN}/args.yaml"
if os.path.exists(args_yaml):
    info["args_yaml"] = yaml.safe_load(open(args_yaml))
json.dump(info, open(f"{OUT_DIR}/baseline_info.json", "w"), indent=2, default=str)
print("Ultralytics now:", info["ultralytics_now"], "| trained with:", info["ultralytics_trained_with"])
print("Best epoch:", info["best_epoch"])
ta = info["train_args_in_checkpoint"] or info.get("args_yaml", {})
for k in ("model", "epochs", "imgsz", "batch", "optimizer", "lr0", "momentum", "weight_decay", "cos_lr",
          "patience", "seed", "deterministic", "amp", "mosaic", "mixup", "fliplr", "flipud",
          "hsv_h", "hsv_s", "hsv_v", "degrees", "scale", "translate", "close_mosaic"):
    print(f"  {k:14s} {ta.get(k)}")

model = YOLO(BEST_PT)
names = {int(k): v for k, v in model.names.items()}
data_yaml = f"{WORK}/data.yaml"
yaml.safe_dump({"path": root, "train": "images/train", "val": "images/val", "names": names},
               open(data_yaml, "w"), sort_keys=False)
print("Classes:", names)

In [ ]:
# ---- 3. Validate at 768 and 640 (standard mAP protocol, conf=0.001) ----
def metrics_dict(r):
    b = r.box
    out = {"mAP50": float(b.map50), "mAP50_95": float(b.map), "P": float(b.mp), "R": float(b.mr), "per_class": {}}
    for i, c in enumerate(b.ap_class_index):
        p, rcl, ap50, ap = b.class_result(i)
        out["per_class"][names[int(c)]] = {"P": float(p), "R": float(rcl), "AP50": float(ap50), "AP50_95": float(ap)}
    out["speed_ms"] = {k: float(v) for k, v in r.speed.items()}
    return out

results = {}
for sz in (768, 640):
    r = model.val(data=data_yaml, imgsz=sz, batch=16, device=DEVICE, split="val", plots=True,
                  project=OUT_DIR, name=f"val_{sz}", exist_ok=True, verbose=False)
    results[f"train768_infer{sz}"] = metrics_dict(r)
    m = results[f"train768_infer{sz}"]
    print(f"\n== Trained 768, evaluated at {sz} ==  mAP50 {m['mAP50']:.3f}  mAP50-95 {m['mAP50_95']:.3f}  P {m['P']:.3f}  R {m['R']:.3f}")
    for c, v in m["per_class"].items():
        print(f"   {c:15s} P {v['P']:.3f}  R {v['R']:.3f}  AP50 {v['AP50']:.3f}  AP50-95 {v['AP50_95']:.3f}")
json.dump(results, open(f"{OUT_DIR}/val_metrics.json", "w"), indent=2)

In [ ]:
# ---- 4. Confusion matrices at conf 0.25 and the deployed 0.60 (input 640, as deployed) ----
import numpy as np
cms = {}
for conf in (0.25, 0.60):
    r = model.val(data=data_yaml, imgsz=640, batch=16, device=DEVICE, split="val", conf=conf, plots=True,
                  project=OUT_DIR, name=f"cm_conf{conf:.2f}", exist_ok=True, verbose=False)
    mat = r.confusion_matrix.matrix.astype(int)          # rows = predicted, cols = true; last = background
    labels = [names[i] for i in range(len(names))] + ["background"]
    cms[f"{conf:.2f}"] = {"labels": labels, "matrix_pred_rows_true_cols": mat.tolist()}
    print(f"\nConfusion matrix at conf {conf:.2f} (rows = predicted, columns = true):")
    print(" " * 16 + "".join(f"{l[:10]:>12s}" for l in labels))
    for i, row in enumerate(mat):
        print(f"{labels[i][:15]:16s}" + "".join(f"{v:12d}" for v in row))
    tp = np.diag(mat)[:-1]; true_tot = mat[:, :-1].sum(0)
    print("  recall per class:", {labels[i]: round(tp[i] / true_tot[i], 3) for i in range(len(tp)) if true_tot[i]})
json.dump(cms, open(f"{OUT_DIR}/confusion_matrices.json", "w"), indent=2)

In [ ]:
# ---- 5. Save every prediction on the val set (conf >= 0.001, input 640) plus ground truth ----
import csv
val_dir = f"{root}/images/val"
files = sorted(f for f in os.listdir(val_dir) if f.lower().endswith(IMG_EXT))
with open(f"{OUT_DIR}/val_predictions.csv", "w", newline="") as fp, open(f"{OUT_DIR}/val_ground_truth.csv", "w", newline="") as fg:
    wp, wg = csv.writer(fp), csv.writer(fg)
    wp.writerow(["image", "cls", "conf", "xc", "yc", "w", "h"])
    wg.writerow(["image", "cls", "xc", "yc", "w", "h"])
    for i in range(0, len(files), 16):
        batch = [f"{val_dir}/{f}" for f in files[i:i + 16]]
        for f, res in zip(files[i:i + 16], model.predict(batch, imgsz=640, conf=0.001, iou=0.7, device=DEVICE, verbose=False)):
            for c, s, b in zip(res.boxes.cls.tolist(), res.boxes.conf.tolist(), res.boxes.xywhn.tolist()):
                wp.writerow([f, int(c), round(s, 5)] + [round(x, 6) for x in b])
    lbl_dir = f"{root}/labels/val"
    for f in files:
        lf = f"{lbl_dir}/{os.path.splitext(f)[0]}.txt"
        if os.path.exists(lf):
            for line in open(lf):
                parts = line.split()
                if len(parts) == 5:
                    wg.writerow([f, int(parts[0])] + parts[1:])
print("Saved val_predictions.csv and val_ground_truth.csv")

In [ ]:
# ---- Summary to paste back into the chat ----
print("=== PASTE EVERYTHING BELOW INTO THE CHAT ===")
summary = {"counts": counts, "baseline": {k: info[k] for k in ("ultralytics_now", "ultralytics_trained_with", "best_epoch")},
           "val": {k: {kk: round(v[kk], 4) for kk in ("mAP50", "mAP50_95", "P", "R")} for k, v in results.items()},
           "per_class_640": {c: {k: round(x, 4) for k, x in d.items()} for c, d in results["train768_infer640"]["per_class"].items()}}
print(json.dumps(summary, indent=1))
for conf, cm in cms.items():
    print(f"confusion_conf{conf} (rows=pred, cols=true, order {cm['labels']}):")
    for row in cm["matrix_pred_rows_true_cols"]:
        print("  ", row)
print("\nAlso upload these 3 files from Drive (fabric/revision/nb1/) into the chat:")
print("  baseline_info.json, val_predictions.csv, val_ground_truth.csv")